In [1]:
import os
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_scheduler
from torch import nn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from tqdm import tqdm

/home/info-sec-lab/BTP/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModel.from_pretrained("../checkpoints/codebert_only/checkpoint-3760")
base_model.to(device)

Some weights of RobertaModel were not initialized from the model checkpoint at ../checkpoints/codebert_only/checkpoint-3760 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(50265, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dr

In [3]:
# Stylometric Features
def extract_stylometric_features(code: str) -> np.ndarray:
    lines = code.splitlines()
    avg_line_length = np.mean([len(line) for line in lines]) if lines else 0
    num_lines = len(lines)
    num_tokens = len(code.split())
    num_chars = len(code)
    return np.array([avg_line_length, num_lines, num_tokens, num_chars], dtype=np.float32)

In [4]:
# Fine-tuning Classifier
class CodeBERTClassifier(nn.Module):
    def __init__(self, base_model, hidden_size=768, num_labels=2):
        super(CodeBERTClassifier, self).__init__()
        self.encoder = base_model
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :]  # CLS token
        logits = self.classifier(pooled_output)
        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)
        return {"logits": logits, "loss": loss}

In [5]:
model = CodeBERTClassifier(base_model)
model.to(device)

CodeBERTClassifier(
  (encoder): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNor

In [6]:
# Dataset for Fine-tuning
class CodeDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512):
        self.data = dataframe.dropna(subset=["code", "label"])
        self.data = self.data[self.data["code"].str.strip().astype(bool)]
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.samples = []
        for _, row in self.data.iterrows():
            tokens = self.tokenizer(row["code"], padding="max_length", truncation=True, max_length=self.max_length)
            if len(tokens["input_ids"]) > 0 and sum(tokens["attention_mask"]) > 0:
                self.samples.append((tokens, row["label"]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        tokens, label = self.samples[idx]
        input_ids = torch.tensor(tokens["input_ids"])
        attention_mask = torch.tensor(tokens["attention_mask"])
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": torch.tensor(label, dtype=torch.long)
        }

In [7]:
valid_df = pd.read_csv("../csvs/val.csv", usecols=["code", "label"])
valid_dataset = CodeDataset(valid_df, tokenizer)
valid_loader = DataLoader(valid_dataset, batch_size=8, shuffle=True)

In [8]:
# Load and Fine-tune
train_df = pd.read_csv("../csvs/train.csv", usecols=["code", "label"])
train_dataset = CodeDataset(train_df, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
num_training_steps = len(train_loader) * 3
lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

# model.train()

In [9]:
for epoch in range(3):
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], labels=batch["labels"])
        loss = outputs["loss"]
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
    print(f"Epoch {epoch+1} loss: {total_loss:.4f}")

Epoch 1:   3%|▎         | 26/940 [00:08<05:04,  3.00it/s]


KeyboardInterrupt: 

In [10]:
# Extract Features
def extract_features(model, dataset):
    loader = DataLoader(dataset, batch_size=8)
    model.eval()
    features = []
    labels = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Extracting Features"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            stylometric = batch["stylometric"].cpu().numpy()
            label_batch = batch["label"]
            outputs = model.encoder(input_ids=input_ids, attention_mask=attention_mask)
            cls_embed = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            combined = np.concatenate([cls_embed, stylometric], axis=1)
            features.append(combined)
            labels.extend(label_batch)
    return np.vstack(features), np.array(labels)

In [12]:
# Build correct EmbeddingDataset, extract features, and train a simple MLP classifier

class EmbeddingDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512, code_col="clean_code"):
        # accept either "clean_code" or "code"
        col = code_col if code_col in dataframe.columns else ("code" if "code" in dataframe.columns else code_col)
        self.data = dataframe.dropna(subset=[col, "label"]).reset_index(drop=True)
        self.data = self.data[self.data[col].str.strip().astype(bool)]
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.code_col = col

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        code = row[self.code_col]
        label = int(row["label"])
        style_feat = extract_stylometric_features(code)
        tokens = self.tokenizer(code, padding="max_length", truncation=True, max_length=self.max_length, return_tensors="pt")
        return {
            "input_ids": tokens["input_ids"].squeeze(0),
            "attention_mask": tokens["attention_mask"].squeeze(0),
            "stylometric": torch.tensor(style_feat, dtype=torch.float32),
            "label": label
        }

# Instantiate embedding datasets using existing dataframes
train_emb_dataset = EmbeddingDataset(train_df, tokenizer)
valid_emb_dataset = EmbeddingDataset(valid_df, tokenizer)

# Extract features (uses model.encoder defined earlier)
X_train, y_train = extract_features(model, train_emb_dataset)
X_val, y_val = extract_features(model, valid_emb_dataset)

# Convert to torch datasets/loaders
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.long)

train_tensor_ds = torch.utils.data.TensorDataset(X_train_t, y_train_t)
val_tensor_ds = torch.utils.data.TensorDataset(X_val_t, y_val_t)

batch_size = 8
train_loader_emb = DataLoader(train_tensor_ds, batch_size=batch_size, shuffle=True)
val_loader_emb = DataLoader(val_tensor_ds, batch_size=batch_size, shuffle=False)

# Simple MLP classifier
class MLPClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, num_labels=2, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_labels)
        )

    def forward(self, x):
        return self.net(x)

input_dim = X_train.shape[1]
clf = MLPClassifier(input_dim=input_dim, hidden_dim=256, num_labels=len(np.unique(y_train)))
clf.to(device)

opt = torch.optim.AdamW(clf.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

# Training loop
num_epochs = 3
for epoch in range(num_epochs):
    clf.train()
    total_loss = 0.0
    for xb, yb in train_loader_emb:
        xb = xb.to(device)
        yb = yb.to(device)
        logits = clf(xb)
        loss = loss_fn(logits, yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss += loss.item() * xb.size(0)
    avg_train_loss = total_loss / len(train_loader_emb.dataset)

    # Validation
    clf.eval()
    preds = []
    trues = []
    with torch.no_grad():
        for xb, yb in val_loader_emb:
            xb = xb.to(device)
            logits = clf(xb)
            preds_batch = torch.argmax(logits, dim=1).cpu().numpy()
            preds.extend(preds_batch.tolist())
            trues.extend(yb.numpy().tolist())

    acc = accuracy_score(trues, preds)
    prec = precision_score(trues, preds, average="binary", zero_division=0) if len(np.unique(y_train))==2 else precision_score(trues, preds, average="macro", zero_division=0)
    rec = recall_score(trues, preds, average="binary", zero_division=0) if len(np.unique(y_train))==2 else recall_score(trues, preds, average="macro", zero_division=0)
    f1 = f1_score(trues, preds, average="binary", zero_division=0) if len(np.unique(y_train))==2 else f1_score(trues, preds, average="macro", zero_division=0)

    print(f"Epoch {epoch+1}/{num_epochs} - train_loss: {avg_train_loss:.4f} val_acc: {acc:.4f} val_prec: {prec:.4f} val_rec: {rec:.4f} val_f1: {f1:.4f}")

# Print final classification report
print("\nValidation classification report:")
print(classification_report(y_val, preds, zero_division=0))

Extracting Features: 100%|██████████| 235/235 [00:23<00:00, 10.17it/s]


Epoch 1/3 - train_loss: 0.4062 val_acc: 0.8367 val_prec: 0.8200 val_rec: 0.8628 val_f1: 0.8409
Epoch 2/3 - train_loss: 0.1168 val_acc: 0.8367 val_prec: 0.8156 val_rec: 0.8702 val_f1: 0.8420
Epoch 3/3 - train_loss: 0.1087 val_acc: 0.8314 val_prec: 0.8228 val_rec: 0.8447 val_f1: 0.8336

Validation classification report:
              precision    recall  f1-score   support

           0       0.84      0.82      0.83       940
           1       0.82      0.84      0.83       940

    accuracy                           0.83      1880
   macro avg       0.83      0.83      0.83      1880
weighted avg       0.83      0.83      0.83      1880

